In [ ]:
!pip install -q transformers accelerate bitsandbytes peft

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("HF_TOKEN")

os.environ['HUGGINGFACE_TOKEN']=huggingface_token

In [ ]:
!huggingface-cli login --token $HUGGINGFACE_TOKEN

In [ ]:
import time
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.data.data_collator import pad_without_fast_tokenizer_warning
import json
import re
from pydantic import BaseModel, Field, ValidationError

torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_flash_sdp(True)

@dataclass
class Config:
    model_name = "google/gemma-2-2b-it"
    max_length = 2048  
    batch_size = 8 
    max_new_tokens = 800  
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()

class NewsArticlePair(BaseModel):
    true_news_title: str = Field(..., min_length=5)
    true_news_body: str = Field(..., min_length=50)
    fake_news_title: str = Field(..., min_length=5)
    fake_news_body: str = Field(..., min_length=50)

print("Loading Gemma-2-2B tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.padding_side = "left"  

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    bnb_8bit_use_double_quant=True,  
)

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Gemma-2-2B model loaded successfully!")

def create_prompt(title, text):
    if pd.isna(title):
        title = "No title"
    if pd.isna(text):
        text = "No content available"
    
    title = str(title)
    text = str(text)
    
    max_input_length = 1500
    if len(text) > max_input_length:
        text = text[:max_input_length] + "..."
    
    prompt = f"""You are a dataset generator for fake news detection research. Generate two article versions from the original.

**Original Article:**
Title: {title}
Content: {text}

**Task:** Create exactly two versions in JSON format:

1. **TRUE NEWS** - Rewrite preserving ALL facts (dates, locations, numbers, names, events). Keep the same meaning.

2. **FAKE NEWS** - Alter specific facts to make it misleading but realistic:
   - Change dates/times (e.g., 2024 → 2023)
   - Change locations (e.g., London → Paris)
   - Modify numbers (e.g., $40 billion → $60 billion)
   - Alter names or attribution
   - Keep professional tone

**Output Format (JSON only, no explanations):**
{{"true_news_title": "rewritten true title", "true_news_body": "rewritten true article preserving all facts", "fake_news_title": "altered fake title", "fake_news_body": "altered fake article with changed facts"}}

Respond with ONLY the JSON object:"""
    
    return prompt

def tokenize_batch(prompts, max_length=cfg.max_length):
    formatted_prompts = []
    for prompt in prompts:
        formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        formatted_prompts.append(formatted)
    
    inputs = tokenizer(
        formatted_prompts,
        max_length=max_length,
        truncation=True,
        padding=False,  
        return_tensors=None,
    )
    
    return inputs["input_ids"], inputs["attention_mask"]

print("Preparing data...")
df = pd.read_csv("/kaggle/input/article/Article.csv")

print(f"Original dataset size: {len(df)}")
df = df.dropna(subset=["title", "main_text"])
print(f"After removing NaN values: {len(df)}")

df = df.reset_index(drop=True)

data = pd.DataFrame()
data["id"] = df.index
data["url"] = df["url"]
data["original_title"] = df["title"]
data["prompt"] = [create_prompt(row["title"], row["main_text"]) for _, row in df.iterrows()]

print("Tokenizing prompts...")
data["input_ids"], data["attention_mask"] = tokenize_batch(data["prompt"].tolist())
data["length"] = data["input_ids"].apply(len)

data = data.sort_values("length", ascending=False).reset_index(drop=True)

print(f"Total articles: {len(data)}")
print(f"Average token length: {data['length'].mean():.0f}")
print(f"Max token length: {data['length'].max()}")

def extract_json_from_response(text):
    text = text.replace("<end_of_turn>", "")
    text = text.replace("<start_of_turn>", "")
    text = text.replace("model\n", "")
    text = re.sub(r'"true_news_title":\s*"rewritten true title"\s*,', '', text)
    text = re.sub(r'"true_news_body":\s*"rewritten true article[^"]*"\s*,', '', text)
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    json_str = text[start:end+1]
    json_str = re.sub(r',(\s*[}\]])', r'\1', json_str)
    
    try:
        data = json.loads(json_str)
        
        for key in ['true_news_title', 'true_news_body', 'fake_news_title', 'fake_news_body']:
            if key in data:
                value = data[key]
                
                if isinstance(value, str):
                    try:
                        decoded = json.loads(value)
                        if isinstance(decoded, (dict, list)):
                            value = str(decoded)
                        else:
                            value = decoded
                    except (json.JSONDecodeError, TypeError):
                        pass
                
                if isinstance(value, (list, dict)):
                    if isinstance(value, list) and all(isinstance(item, dict) for item in value):
                        parts = []
                        for item in value:
                            part = ', '.join(f"{k}: {v}" for k, v in item.items())
                            parts.append(part)
                        data[key] = '. '.join(parts) + '.'
                    else:
                        data[key] = str(value)
                elif not isinstance(value, str):
                    data[key] = str(value)
                
                if isinstance(data[key], str):
                    data[key] = data[key].replace('\\n', '\n').replace('\\t', ' ')
                    data[key] = re.sub(r'\n+', '. ', data[key])
                    data[key] = re.sub(r'\s+', ' ', data[key]).strip()
        
        required_fields = ['true_news_title', 'true_news_body', 'fake_news_title', 'fake_news_body']
        if not all(key in data and isinstance(data[key], str) and len(data[key]) > 0 
                   for key in required_fields):
            return None
        
        if (data['true_news_title'] == data['fake_news_title'] and 
            data['true_news_body'] == data['fake_news_body']):
            return None
            
        return NewsArticlePair(**data)
    except (json.JSONDecodeError, ValidationError, TypeError, KeyError) as e:
        return None

@torch.no_grad()
@torch.amp.autocast('cuda')
def inference_batch(batch_df, batch_size=cfg.batch_size, max_new_tokens=cfg.max_new_tokens, debug=False):
    results = []
    
    pbar = tqdm(range(0, len(batch_df), batch_size), desc="Batch progress", leave=False)
    
    for start_idx in pbar:
        end_idx = min(start_idx + batch_size, len(batch_df))
        batch = batch_df.iloc[start_idx:end_idx]
        input_ids = batch["input_ids"].tolist()
        attention_mask = batch["attention_mask"].tolist()
        
        inputs = pad_without_fast_tokenizer_warning(
            tokenizer,
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding="longest",
            pad_to_multiple_of=None,
            return_tensors="pt",
        )
        
        outputs = model.generate(
            **inputs.to(cfg.device),
            max_new_tokens=max_new_tokens,
            temperature=0.6,  
            top_p=0.95,
            top_k=40,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            repetition_penalty=1.15,  
        )
        
        for i, output in enumerate(outputs):
            input_length = len(input_ids[i])
            generated_ids = output[input_length:]
            generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            
            if debug and start_idx == 0 and i < 2:
                print(f"\n{'='*80}")
                print(f"DEBUG - Sample Response {i+1}:")
                print(f"{'='*80}")
                print(generated_text[:500])
                print(f"{'='*80}\n")
            
            parsed = extract_json_from_response(generated_text)
            
            if parsed:
                results.append({
                    "id": batch.iloc[i]["id"],
                    "url": batch.iloc[i]["url"],
                    "original_title": batch.iloc[i]["original_title"],
                    "true_news_title": parsed.true_news_title,
                    "true_news_body": parsed.true_news_body,
                    "fake_news_title": parsed.fake_news_title,
                    "fake_news_body": parsed.fake_news_body,
                })
            else:
                if debug and start_idx == 0 and i < 2:
                    print(f"FAILED TO PARSE RESPONSE {i+1}")
                    print(f"Looking for JSON in: {generated_text[:300]}...")
                results.append(None)
        
        success_rate = (len([r for r in results if r is not None]) / len(results) * 100) if results else 0
        pbar.set_postfix({"success_rate": f"{success_rate:.1f}%", "processed": len(results)})
    
    pbar.close()
    return results

print(f"\nStarting inference with batch_size={cfg.batch_size}...")
st = time.time()

all_results = []
failed_indices = []

chunk_size = 1000
total_chunks = (len(data) + chunk_size - 1) // chunk_size

chunk_pbar = tqdm(range(0, len(data), chunk_size), desc="Overall progress", unit="chunk")

for chunk_start in chunk_pbar:
    chunk_end = min(chunk_start + chunk_size, len(data))
    chunk_df = data.iloc[chunk_start:chunk_end]
    chunk_num = chunk_start // chunk_size + 1
    
    chunk_pbar.set_description(f"Chunk {chunk_num}/{total_chunks}")
    
    chunk_st = time.time()
    chunk_results = inference_batch(chunk_df, batch_size=cfg.batch_size, debug=(chunk_num == 1))
    chunk_time = time.time() - chunk_st
    
    for i, result in enumerate(chunk_results):
        idx = chunk_start + i
        if result:
            all_results.append(result)
        else:
            failed_indices.append(idx)
    
    chunk_success = len([r for r in chunk_results if r is not None])
    chunk_success_rate = chunk_success / len(chunk_results) * 100
    overall_success_rate = len(all_results) / (chunk_end) * 100
    
    elapsed = time.time() - st
    avg_time_per_article = elapsed / chunk_end
    remaining_articles = len(data) - chunk_end
    estimated_remaining = avg_time_per_article * remaining_articles
    
    chunk_pbar.set_postfix({
        "chunk_time": f"{chunk_time/60:.1f}min",
        "chunk_success": f"{chunk_success_rate:.1f}%",
        "overall_success": f"{overall_success_rate:.1f}%",
        "eta": f"{estimated_remaining/60:.1f}min"
    })
    
    if all_results:
        checkpoint_df = pd.DataFrame(all_results)
        checkpoint_df.to_parquet(f"checkpoint_{chunk_end}.parquet", index=False)

chunk_pbar.close()

elapsed_time = time.time() - st
print(f"\nTotal inference time: {elapsed_time/60:.1f} minutes ({elapsed_time/3600:.2f} hours)")
print(f"Average time per article: {elapsed_time/len(data):.2f} seconds")
print(f"Throughput: {len(data)/(elapsed_time/3600):.0f} articles/hour")

print("\nSaving final results...")
result_df = pd.DataFrame(all_results)
result_df.to_parquet("augmented_fake_true_news_gemma2.parquet", index=False)
result_df.to_csv("augmented_fake_true_news_gemma2.csv", index=False)

print(f"\n{'='*60}")
print(f"Completed! Successfully processed {len(all_results)}/{len(data)} articles")
print(f"Success rate: {len(all_results)/len(data)*100:.1f}%")
print(f"Failed: {len(failed_indices)} articles")
print(f"{'='*60}")

if failed_indices:
    failed_df = pd.DataFrame({
        "failed_index": failed_indices,
        "title": [data.iloc[i]["original_title"] for i in failed_indices]
    })
    failed_df.to_csv("failed_indices.csv", index=False)

# Show Sample
if len(all_results) > 0:
    print("\n" + "="*60)
    print("SAMPLE OUTPUT:")
    print("="*60)
    sample = all_results[0]
    print(f"\nOriginal Title: {sample['original_title']}")
    print(f"\nTrue News Title: {sample['true_news_title']}")
    print(f"True News Body: {sample['true_news_body'][:200]}...")
    print(f"\nFake News Title: {sample['fake_news_title']}")
    print(f"Fake News Body: {sample['fake_news_body'][:200]}...")